# NYC Restaurant Recommendation Agent

This Colab notebook uses:
- a New York restaurant CSV as the main dataset
- Google Places API for live place enrichment
- Gemini via Google AI Studio for the final recommendation response

Built-in guardrails:
- asks whether to use current location or a manually entered location
- validates that the location is in New York City
- keeps recommendations grounded in the dataset and API results


In [ ]:
%pip install -q -U pandas requests google-genai

In [ ]:
import math
import os
import re
from getpass import getpass

import pandas as pd
import requests
from google import genai

DATA_PATH = "trip advisor restaurents  10k - trip_rest_neywork_1.csv"
GEMINI_MODEL = "gemini-2.0-flash"
NYC_BOUNDS = {
    "lat_min": 40.45,
    "lat_max": 40.95,
    "lng_min": -74.30,
    "lng_max": -73.65,
}

pd.set_option("display.max_colwidth", 160)

## Enter API keys

Use your Google AI Studio key for Gemini and your Google Maps Platform key for Places.

If you already stored them as environment variables in Colab, the notebook will reuse them:
- `GEMINI_API_KEY`
- `GOOGLE_MAPS_API_KEY`


In [ ]:
GEMINI_API_KEY = os.getenv("GEMINI_API_KEY") or getpass("Enter GEMINI_API_KEY: ")
GOOGLE_MAPS_API_KEY = os.getenv("GOOGLE_MAPS_API_KEY") or getpass("Enter GOOGLE_MAPS_API_KEY: ")

gemini_client = genai.Client(api_key=GEMINI_API_KEY)
print("Keys captured. Gemini client is ready.")

In [ ]:
df = pd.read_csv(DATA_PATH).copy()

df = df.rename(
    columns={
        "Title": "title",
        "Number of review": "review_count_raw",
        "Catagory": "category",
        "Reveiw Comment": "review_comment",
        "Popular food": "popular_food",
        "Online Order": "online_order",
    }
)

for col in ["title", "category", "review_comment", "popular_food", "online_order"]:
    df[col] = df[col].fillna("").astype(str).str.strip()

df["review_count"] = (
    df["review_count_raw"]
    .astype(str)
    .str.replace(",", "", regex=False)
    .str.extract(r"(\d+)")
    .astype(float)
    .fillna(0)
    .astype(int)
)

df["category_list"] = df["category"].apply(
    lambda value: [part.strip() for part in str(value).split(",") if part.strip()]
)
df["search_blob"] = (
    df["title"]
    + " "
    + df["category"]
    + " "
    + df["review_comment"]
    + " "
    + df["popular_food"]
).str.lower()

df = df.drop_duplicates(subset=["title", "category", "popular_food", "review_comment"]).reset_index(drop=True)
df.head()

In [ ]:
def normalize_text(value: str) -> str:
    value = str(value or "").lower().strip()
    value = re.sub(r"[^a-z0-9,\s-]", " ", value)
    value = re.sub(r"\s+", " ", value)
    return value.strip()


def split_terms(value: str):
    cleaned = normalize_text(value)
    if not cleaned:
        return []
    parts = re.split(r",|/|\band\b", cleaned)
    return [part.strip() for part in parts if part.strip()]


def is_in_nyc(lat: float, lng: float) -> bool:
    return (
        NYC_BOUNDS["lat_min"] <= lat <= NYC_BOUNDS["lat_max"]
        and NYC_BOUNDS["lng_min"] <= lng <= NYC_BOUNDS["lng_max"]
    )


def haversine_miles(lat1: float, lng1: float, lat2: float, lng2: float) -> float:
    radius_miles = 3958.8
    d_lat = math.radians(lat2 - lat1)
    d_lng = math.radians(lng2 - lng1)
    a = (
        math.sin(d_lat / 2) ** 2
        + math.cos(math.radians(lat1))
        * math.cos(math.radians(lat2))
        * math.sin(d_lng / 2) ** 2
    )
    return 2 * radius_miles * math.asin(math.sqrt(a))


def places_text_search(text_query: str, field_mask: str, max_result_count: int = 5, location_bias=None):
    url = "https://places.googleapis.com/v1/places:searchText"
    headers = {
        "Content-Type": "application/json",
        "X-Goog-Api-Key": GOOGLE_MAPS_API_KEY,
        "X-Goog-FieldMask": field_mask,
    }
    payload = {
        "textQuery": text_query,
        "maxResultCount": max_result_count,
    }
    if location_bias:
        payload["locationBias"] = location_bias
    response = requests.post(url, headers=headers, json=payload, timeout=30)
    response.raise_for_status()
    return response.json().get("places", [])


def geocode_nyc_location(location_text: str):
    query = f"{location_text}, New York City"
    places = places_text_search(
        text_query=query,
        field_mask="places.displayName,places.formattedAddress,places.location",
        max_result_count=1,
    )
    if not places:
        raise ValueError("Location not found. Please enter a New York City neighborhood, address, or ZIP code.")

    place = places[0]
    lat = place["location"]["latitude"]
    lng = place["location"]["longitude"]
    if not is_in_nyc(lat, lng):
        raise ValueError("The entered location is outside New York City. Please enter an NYC location.")

    return {
        "label": place.get("displayName", {}).get("text", location_text),
        "address": place.get("formattedAddress", location_text),
        "lat": lat,
        "lng": lng,
        "source": "manual",
    }


def get_browser_location():
    try:
        from google.colab import output
    except ImportError as exc:
        raise RuntimeError("Browser location works only in Google Colab. Use manual location instead.") from exc

    js = """
    async function getLocation() {
      return await new Promise((resolve, reject) => {
        navigator.geolocation.getCurrentPosition(
          (position) => resolve({lat: position.coords.latitude, lng: position.coords.longitude}),
          (error) => reject(error.message),
          {enableHighAccuracy: true, timeout: 10000}
        );
      });
    }
    """
    coords = output.eval_js(js + "getLocation()")
    lat = float(coords["lat"])
    lng = float(coords["lng"])
    if not is_in_nyc(lat, lng):
        raise ValueError("Current location is outside New York City. Please switch to manual NYC location.")
    return {
        "label": "Current browser location",
        "address": "Current NYC location",
        "lat": lat,
        "lng": lng,
        "source": "current",
    }


def collect_user_location():
    print("Location guardrail: choose how to set the location.")
    choice = input("Type 'current' to use current location or 'manual' to enter a New York location: ").strip().lower()
    if choice == "current":
        return get_browser_location()
    if choice == "manual":
        manual_location = input("Enter a New York City location, neighborhood, ZIP code, or address: ").strip()
        return geocode_nyc_location(manual_location)
    raise ValueError("Please type exactly 'current' or 'manual'.")


def score_dataset_match(row, cuisine_terms, food_terms, ambience_terms, online_order_preference):
    score = 0.0
    blob = row["search_blob"]
    category_text = normalize_text(row["category"])
    popular_food = normalize_text(row["popular_food"])
    online_order = normalize_text(row["online_order"])

    if cuisine_terms:
        matches = sum(term in category_text or term in blob for term in cuisine_terms)
        score += matches * 4.0

    if food_terms:
        matches = sum(term in popular_food or term in blob for term in food_terms)
        score += matches * 3.5

    if ambience_terms:
        matches = sum(term in blob for term in ambience_terms)
        score += matches * 2.5

    if online_order_preference == "yes" and online_order == "yes":
        score += 1.5
    elif online_order_preference == "no" and online_order == "no":
        score += 1.0

    score += min(row["review_count"], 5000) / 1000
    return score


def get_candidate_matches(dataframe, cuisine, food, ambience, online_order, limit=20):
    cuisine_terms = split_terms(cuisine)
    food_terms = split_terms(food)
    ambience_terms = split_terms(ambience)
    online_order_preference = normalize_text(online_order)

    ranked = dataframe.copy()
    ranked["dataset_score"] = ranked.apply(
        lambda row: score_dataset_match(row, cuisine_terms, food_terms, ambience_terms, online_order_preference),
        axis=1,
    )

    if cuisine_terms or food_terms or ambience_terms:
        ranked = ranked[ranked["dataset_score"] > 0]

    ranked = ranked.sort_values(["dataset_score", "review_count"], ascending=[False, False])
    return ranked.head(limit).reset_index(drop=True)


def enrich_with_places(candidates, user_location, min_google_rating=0.0):
    rows = []
    location_bias = {
        "circle": {
            "center": {
                "latitude": user_location["lat"],
                "longitude": user_location["lng"],
            },
            "radius": 15000.0,
        }
    }

    for row in candidates.itertuples(index=False):
        query = f"{row.title} restaurant in New York City"
        places = places_text_search(
            text_query=query,
            field_mask=(
                "places.displayName,places.formattedAddress,places.location,"
                "places.rating,places.userRatingCount,places.googleMapsUri,places.primaryType"
            ),
            max_result_count=1,
            location_bias=location_bias,
        )

        place = places[0] if places else {}
        place_location = place.get("location", {})
        lat = place_location.get("latitude")
        lng = place_location.get("longitude")
        distance_miles = None
        if lat is not None and lng is not None:
            distance_miles = round(haversine_miles(user_location["lat"], user_location["lng"], lat, lng), 2)

        google_rating = place.get("rating")
        if google_rating is not None and google_rating < min_google_rating:
            continue

        hybrid_score = float(row.dataset_score)
        if google_rating is not None:
            hybrid_score += google_rating * 2.0
        if distance_miles is not None:
            hybrid_score += max(0, 5 - min(distance_miles, 5))

        rows.append(
            {
                "title": row.title,
                "category": row.category,
                "review_count": int(row.review_count),
                "review_comment": row.review_comment,
                "popular_food": row.popular_food,
                "online_order": row.online_order,
                "dataset_score": round(float(row.dataset_score), 2),
                "google_name": place.get("displayName", {}).get("text"),
                "google_address": place.get("formattedAddress"),
                "google_rating": google_rating,
                "google_user_ratings_total": place.get("userRatingCount"),
                "distance_miles": distance_miles,
                "google_maps_url": place.get("googleMapsUri"),
                "hybrid_score": round(hybrid_score, 2),
                "review_source": "TripAdvisor CSV",
                "live_place_source": "Google Places API",
            }
        )

    if not rows:
        raise ValueError("No matching restaurants were found after Google Places enrichment. Try relaxing the filters.")

    result_df = pd.DataFrame(rows)
    result_df = result_df.sort_values(["hybrid_score", "review_count"], ascending=[False, False]).reset_index(drop=True)
    return result_df


def build_prompt(user_location, cuisine, food, ambience, online_order, min_google_rating, top_results):
    result_records = top_results.to_dict(orient="records")
    prompt_lines = [
        "You are an NYC restaurant recommendation assistant.",
        "",
        "Instructions:",
        "- Use only the restaurant information provided below.",
        "- Do not invent ratings, addresses, ambience, or review sources.",
        "- If ambience is not directly stated, say it is inferred from review text.",
        "- Mention that the review snippet source is TripAdvisor CSV data and the live place metadata source is Google Places API.",
        "- Recommend 3 restaurants maximum.",
        "- For each recommendation, explain why it fits the user request.",
        "- End with one short note if the data has a limitation.",
        "",
        "User preferences:",
        f"- Location: {user_location}",
        f"- Cuisine: {cuisine or 'No cuisine preference provided'}",
        f"- Food or dish: {food or 'No specific dish provided'}",
        f"- Ambience: {ambience or 'No ambience preference provided'}",
        f"- Online order preference: {online_order or 'No preference provided'}",
        f"- Minimum Google rating: {min_google_rating}",
        "",
        "Candidate restaurants:",
        str(result_records),
    ]
    return "\n".join(prompt_lines)


def ask_gemini_for_recommendations(user_location, cuisine, food, ambience, online_order, min_google_rating, top_results):
    prompt = build_prompt(user_location, cuisine, food, ambience, online_order, min_google_rating, top_results)
    response = gemini_client.models.generate_content(
        model=GEMINI_MODEL,
        contents=prompt,
    )
    return response.text

## Run the agent

The agent asks for the location mode first, which is part of the guardrail.

Suggested inputs:
- cuisine: `Italian`
- food or dish: `pizza`
- ambience: `cozy, date night`
- online order: `yes` or leave blank
- minimum Google rating: `4.2` or leave blank


In [ ]:
user_location = collect_user_location()
print("Resolved user location:", user_location)

cuisine = input("Cuisine preference: ").strip()
food = input("Food or dish preference: ").strip()
ambience = input("Ambience preference: ").strip()
online_order = input("Need online order? Type yes, no, or leave blank: ").strip()
min_google_rating_raw = input("Minimum Google rating (press Enter to skip): ").strip()
min_google_rating = float(min_google_rating_raw) if min_google_rating_raw else 0.0

candidate_matches = get_candidate_matches(
    dataframe=df,
    cuisine=cuisine,
    food=food,
    ambience=ambience,
    online_order=online_order,
    limit=15,
)

enriched_results = enrich_with_places(
    candidates=candidate_matches,
    user_location=user_location,
    min_google_rating=min_google_rating,
)

display(enriched_results.head(10))

final_answer = ask_gemini_for_recommendations(
    user_location=user_location,
    cuisine=cuisine,
    food=food,
    ambience=ambience,
    online_order=online_order,
    min_google_rating=min_google_rating,
    top_results=enriched_results.head(5),
)

print(final_answer)

## Notes

- This CSV does not include exact restaurant coordinates, so distance comes from Google Places enrichment.
- Ambience is inferred mainly from review text because the dataset does not have a dedicated ambience column.
- Review source in this notebook is the TripAdvisor CSV, while live ratings and addresses come from Google Places.
- If you want, the next improvement would be adding a small `ipywidgets` UI so the notebook feels more like a mini app.
